# 03 — GDC API: sanity check klienta

Notebook do **operacyjnej weryfikacji klienta GDC API** zaimplementowanego w trzech ostatnich commitach:
- `gdc_client.py` — endpoint `/files` (zapytania + pobieranie plików z MD5)
- `cases_client.py` — endpoint `/cases` (dane kliniczne pacjentów)

W odróżnieniu od notebooków 01 i 02 (sanity check biologiczny i operacyjny pełnej kohorty), ten notebook **NIE używa lokalnych plików w `data/raw/`** — wszystko ściąga z żywego API portalu Genomic Data Commons. Idealny do:

1. Weryfikacji że API jest dostępne i działa
2. Sprawdzenia że klient zwraca dane w spójnym formacie
3. Testowania kompletnego cyklu: query → parse → save → parse_clinical/parse_star_counts
4. Pomiaru wydajności pobierania (czas, throughput)

**Uwaga:** notebook pobiera tylko 3-5 testowych plików (~15 MB), nie całą kohortę. Bezpieczny do odpalania wielokrotnego.


In [1]:
import sys
from pathlib import Path
from datetime import datetime, timezone
import tempfile

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingest import (
    build_files_filter,
    query_files,
    parse_files_response,
    download_files,
    build_cases_filter,
    query_cases,
    parse_cases_response,
    parse_clinical,
    parse_star_counts,
    GDCClientError,
    CasesClientError,
)

print(f"Projekt: {PROJECT_ROOT}")
print(f"Polars: {pl.__version__}")
print(f"Czas startu: {datetime.now(timezone.utc).isoformat()}")


Projekt: /Users/luka/luad-huba-clean
Polars: 1.40.1
Czas startu: 2026-06-07T09:00:28.255252+00:00


## 1. Test endpointu `/files` — zapytanie + parsowanie

Najprostsze możliwe wywołanie: filtr domyślny (TCGA-LUAD + STAR Counts), 5 plików, parsowanie do DataFrame. Sprawdza:
- czy API odpowiada
- czy filtr zwraca to czego się spodziewamy (RNA-seq STAR Counts)
- czy parser radzi sobie z zagnieżdżoną strukturą `cases > samples > portions > analytes > aliquots`


In [2]:
print("=== Zapytanie do /files endpoint ===")
filt = build_files_filter()
print(f"Filtr: project={filt['content'][0]['content']['value'][0]}, "
      f"workflow={filt['content'][1]['content']['value'][0]}")
print()

response = query_files(filters=filt, size=5)

n_hits = len(response["data"]["hits"])
total_available = response["data"]["pagination"]["total"]
print(f"Otrzymano: {n_hits} plików (z {total_available} dostępnych w GDC dla tego filtra)")


=== Zapytanie do /files endpoint ===
Filtr: project=TCGA-LUAD, workflow=STAR - Counts

Otrzymano: 5 plików (z 601 dostępnych w GDC dla tego filtra)


In [3]:
print("=== Parsowanie odpowiedzi ===")
files_df = parse_files_response(response)
print(f"DataFrame: {files_df.height} wierszy x {files_df.width} kolumn")
print()
print("Kolumny:")
for c in files_df.columns:
    print(f"  - {c}")
print()
print("Pierwszy rekord:")
for col in files_df.columns:
    val = files_df[col][0]
    if isinstance(val, str) and len(val) > 50:
        val = val[:47] + "..."
    print(f"  {col:30} {val}")


=== Parsowanie odpowiedzi ===
DataFrame: 5 wierszy x 13 kolumn

Kolumny:
  - file_id
  - file_name
  - md5sum
  - file_size
  - data_type
  - experimental_strategy
  - sample_id
  - case_submitter_id
  - aliquot_barcode
  - case_uuid
  - aliquot_uuid
  - workflow_type
  - workflow_version

Pierwszy rekord:
  file_id                        140633c2-a8e9-4a2d-a044-082cd099ff98
  file_name                      b7f29b8c-08a8-4781-ba33-5a7b6b02ad23.rna_seq.au...
  md5sum                         62ab2e7a471de153bbf4530d13914cd6
  file_size                      4232632
  data_type                      Gene Expression Quantification
  experimental_strategy          RNA-Seq
  sample_id                      TCGA-NJ-A4YQ-01A
  case_submitter_id              TCGA-NJ-A4YQ
  aliquot_barcode                TCGA-NJ-A4YQ-01A-11R-A262-07
  case_uuid                      52df074f-a402-4b78-9472-f8eb268efded
  aliquot_uuid                   72cad61d-c9cf-4507-b79a-9ecf459a119d
  workflow_type             

## 2. Sanity checks na odpowiedzi `/files`

Weryfikacja, że dane mają sens biologicznie i strukturalnie.


In [4]:
print("=== Workflow type - powinno być wszędzie 'STAR - Counts' ===")
print(files_df.group_by("workflow_type").len())

print()
print("=== Workflow version - powinien być TEN SAM hash gita ===")
print(files_df["workflow_version"].unique())
print()
print("(Jeśli >1 wartość, znaczy że GDC re-processuje kohortę różnymi wersjami)")

print()
print("=== Rozmiary plików ===")
size_stats = files_df.select([
    pl.col("file_size").min().alias("min_bytes"),
    pl.col("file_size").max().alias("max_bytes"),
    pl.col("file_size").mean().alias("mean_bytes"),
])
print(size_stats)
print(f"  ~{files_df['file_size'].mean() / 1024**2:.1f} MB średnio (typowo ~4 MB dla STAR Counts)")

print()
print("=== Sample IDs ===")
print(files_df.select(["sample_id", "case_submitter_id", "aliquot_barcode"]))


=== Workflow type - powinno być wszędzie 'STAR - Counts' ===
shape: (1, 2)
┌───────────────┬─────┐
│ workflow_type ┆ len │
│ ---           ┆ --- │
│ str           ┆ u32 │
╞═══════════════╪═════╡
│ STAR - Counts ┆ 5   │
└───────────────┴─────┘

=== Workflow version - powinien być TEN SAM hash gita ===
shape: (1,)
Series: 'workflow_version' [str]
[
	"eecca6e2e475aab335bd7c365025ca…
]

(Jeśli >1 wartość, znaczy że GDC re-processuje kohortę różnymi wersjami)

=== Rozmiary plików ===
shape: (1, 3)
┌───────────┬───────────┬────────────┐
│ min_bytes ┆ max_bytes ┆ mean_bytes │
│ ---       ┆ ---       ┆ ---        │
│ i64       ┆ i64       ┆ f64        │
╞═══════════╪═══════════╪════════════╡
│ 4232632   ┆ 4251071   ┆ 4241591.6  │
└───────────┴───────────┴────────────┘
  ~4.0 MB średnio (typowo ~4 MB dla STAR Counts)

=== Sample IDs ===
shape: (5, 3)
┌──────────────────┬───────────────────┬──────────────────────────────┐
│ sample_id        ┆ case_submitter_id ┆ aliquot_barcode              │
│ 

## 3. Test endpointu `/cases` — dane kliniczne

Drugie zapytanie do innego endpointu — `/cases` zamiast `/files`. To inny model danych (zorientowany na pacjenta, nie na plik). Sprawdza klienta `cases_client.py`.


In [5]:
print("=== Zapytanie do /cases endpoint ===")
filt_cases = build_cases_filter()
print(f"Filtr: project={filt_cases['content']['value'][0]}")
print()

response_cases = query_cases(filters=filt_cases, size=5)

n_cases = len(response_cases["data"]["hits"])
total_cases = response_cases["data"]["pagination"]["total"]
print(f"Otrzymano: {n_cases} pacjentów (z {total_cases} dostępnych dla TCGA-LUAD)")


=== Zapytanie do /cases endpoint ===
Filtr: project=TCGA-LUAD

Otrzymano: 5 pacjentów (z 585 dostępnych dla TCGA-LUAD)


In [6]:
print("=== Parsowanie odpowiedzi /cases ===")
cases_df = parse_cases_response(response_cases)
print(f"DataFrame: {cases_df.height} wierszy x {cases_df.width} kolumn")
print()
print("Pierwsze wiersze (format identyczny z clinical.tsv z portalu):")
print(cases_df)


=== Parsowanie odpowiedzi /cases ===
DataFrame: 15 wierszy x 10 kolumn

Pierwsze wiersze (format identyczny z clinical.tsv z portalu):
shape: (15, 10)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ cases.sub ┆ cases.pri ┆ demograph ┆ demograph ┆ … ┆ demograph ┆ diagnoses ┆ diagnoses ┆ diagnose │
│ mitter_id ┆ mary_site ┆ ic.vital_ ┆ ic.days_t ┆   ┆ ic.race   ┆ .diagnosi ┆ .days_to_ ┆ s.ajcc_p │
│ ---       ┆ ---       ┆ status    ┆ o_death   ┆   ┆ ---       ┆ s_is_prim ┆ last_foll ┆ athologi │
│ str       ┆ str       ┆ ---       ┆ ---       ┆   ┆ str       ┆ ary…      ┆ ow_…      ┆ c_stag…  │
│           ┆           ┆ str       ┆ str       ┆   ┆           ┆ ---       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆ str       ┆ str       ┆ str      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ TCGA-44-2 ┆ Bronchus  ┆ Alive     ┆    

## 4. KRYTYCZNY test integracji — `query_cases → TSV → parse_clinical`

Najważniejszy test architektoniczny: czy wynik `parse_cases_response` zapisany jako TSV jest pełnoprawnym wejściem dla istniejącego `parse_clinical`? Jeśli tak, pipeline jest naprawdę samowystarczalny — clinical.tsv z portalu i z API są zamienne.


In [7]:
print("=== Zapis cases_df jako TSV (format jak z portalu) ===")
tsv_path = Path(tempfile.mktemp(suffix="_clinical.tsv"))
cases_df.write_csv(tsv_path, separator="\t", quote_style="never")
print(f"Zapisano: {tsv_path}")
print()
print("Pierwsze linie:")
print(tsv_path.read_text()[:600])


=== Zapis cases_df jako TSV (format jak z portalu) ===
Zapisano: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/tmpnx84qfky_clinical.tsv

Pierwsze linie:
cases.submitter_id	cases.primary_site	demographic.vital_status	demographic.days_to_death	demographic.age_at_index	demographic.gender	demographic.race	diagnoses.diagnosis_is_primary_disease	diagnoses.days_to_last_follow_up	diagnoses.ajcc_pathologic_stage
TCGA-44-2655	Bronchus and lung	Alive		65	female	white	true	1324.0	Stage IA
TCGA-44-2655	Bronchus and lung	Alive		65	female	white	false		
TCGA-44-2655	Bronchus and lung	Alive		65	female	white	false		
TCGA-44-7671	Bronchus and lung	Alive		64	male	black or african american	true	889.0	Stage IB
TCGA-44-7671	Bronchus and lung	Alive		64	male	black or 


In [8]:
print("=== Parsowanie przez parse_clinical ===")
clinical_df = parse_clinical(tsv_path)
print(f"Wynik: {clinical_df.height} unikalnych pacjentów x {clinical_df.width} kolumn")
print()
print("Kolumny (z obliczonymi time i event):")
print(clinical_df.columns)
print()
print("Wyciąg najważniejszych kolumn:")
print(clinical_df.select([
    "case_submitter_id",
    "vital_status",
    "time",
    "event",
    "age_at_index",
    "gender",
    "ajcc_pathologic_stage",
]))


=== Parsowanie przez parse_clinical ===
Wynik: 5 unikalnych pacjentów x 11 kolumn

Kolumny (z obliczonymi time i event):
['case_submitter_id', 'vital_status', 'time', 'event', 'days_to_death', 'days_to_last_follow_up', 'age_at_index', 'gender', 'race', 'ajcc_pathologic_stage', 'primary_site']

Wyciąg najważniejszych kolumn:
shape: (5, 7)
┌───────────────────┬──────────────┬──────┬───────┬──────────────┬────────┬───────────────────────┐
│ case_submitter_id ┆ vital_status ┆ time ┆ event ┆ age_at_index ┆ gender ┆ ajcc_pathologic_stage │
│ ---               ┆ ---          ┆ ---  ┆ ---   ┆ ---          ┆ ---    ┆ ---                   │
│ str               ┆ str          ┆ i64  ┆ bool  ┆ i64          ┆ str    ┆ str                   │
╞═══════════════════╪══════════════╪══════╪═══════╪══════════════╪════════╪═══════════════════════╡
│ TCGA-44-2655      ┆ Alive        ┆ 1324 ┆ false ┆ 65           ┆ female ┆ Stage IA              │
│ TCGA-44-7671      ┆ Alive        ┆ 889  ┆ false ┆ 64      

**Jeśli powyższe działa bez błędu i pokazuje sensowne time/event** — pełna pętla API → clinical działa. Pipeline jest niezależny od ręcznego pobierania `clinical.tsv` z portalu.

## 5. Test pobierania plików — `download_files` z weryfikacją MD5

Trzy pliki, ~12 MB, do katalogu tymczasowego (nie `data/raw/`). Sprawdza:
- pobieranie przez streaming
- obliczanie MD5 incrementalnie
- weryfikację z `md5sum` z `parse_files_response`
- pasek postępu tqdm


In [9]:
print("=== Pobieranie 3 plików testowych ===")
download_dir = Path(tempfile.mkdtemp(prefix="luad_huba_test_"))
print(f"Katalog: {download_dir}")
print()

metadata = files_df.head(3)
print(f"Łączny rozmiar do pobrania: {metadata['file_size'].sum() / 1024**2:.1f} MB")
print()

result = download_files(
    metadata=metadata,
    output_dir=download_dir,
    show_progress=True,
)


=== Pobieranie 3 plików testowych ===
Katalog: /var/folders/69/yp9cyjqn07g6xrjh38nxblg80000gn/T/luad_huba_test_kjywwlni

Łączny rozmiar do pobrania: 12.1 MB



Pobieranie z GDC: 100%|█████████████████████████| 3/3 [00:12<00:00,  4.31s/plik]


In [10]:
print("=== Raport pobierania ===")
print(result.select([
    "file_id",
    "verified",
    "bytes_downloaded",
    "duration_s",
    "attempts",
    "error",
]))

print()
print("=== Sumaryczne ===")
total_bytes = result["bytes_downloaded"].sum()
total_time = result["duration_s"].sum()
n_verified = result.filter(pl.col("verified")).height
n_failed = result.filter(~pl.col("verified")).height

print(f"Pobrano: {total_bytes / 1024**2:.1f} MB w {total_time:.1f}s")
if total_time > 0:
    print(f"Średnia przepustowość: {total_bytes / 1024**2 / total_time:.2f} MB/s")
print(f"Zweryfikowane: {n_verified}/{result.height}")
print(f"Błędy: {n_failed}")


=== Raport pobierania ===
shape: (3, 6)
┌─────────────────────────────────┬──────────┬──────────────────┬────────────┬──────────┬───────┐
│ file_id                         ┆ verified ┆ bytes_downloaded ┆ duration_s ┆ attempts ┆ error │
│ ---                             ┆ ---      ┆ ---              ┆ ---        ┆ ---      ┆ ---   │
│ str                             ┆ bool     ┆ i64              ┆ f64        ┆ i64      ┆ str   │
╞═════════════════════════════════╪══════════╪══════════════════╪════════════╪══════════╪═══════╡
│ 140633c2-a8e9-4a2d-a044-082cd0… ┆ true     ┆ 4232632          ┆ 4.238418   ┆ 1        ┆       │
│ 15a57d06-841c-473e-b89e-389f1d… ┆ true     ┆ 4239650          ┆ 4.725237   ┆ 1        ┆       │
│ 5ecec8fc-845f-43cf-82c3-73dea8… ┆ true     ┆ 4235780          ┆ 3.944931   ┆ 1        ┆       │
└─────────────────────────────────┴──────────┴──────────────────┴────────────┴──────────┴───────┘

=== Sumaryczne ===
Pobrano: 12.1 MB w 12.9s
Średnia przepustowość: 0.94 MB/s


## 6. Test idempotentności — drugie uruchomienie

Kluczowa właściwość: ponowne wywołanie `download_files` na tym samym katalogu **nie powinno** pobierać plików, które już są lokalnie z poprawnym MD5. Pozwala wznawiać przerwane pobieranie kohorty.


In [11]:
print("=== Drugie uruchomienie download_files (skip_existing=True default) ===")
result_second = download_files(
    metadata=metadata,
    output_dir=download_dir,
    show_progress=True,
)

print()
print("=== Drugi raport ===")
print(result_second.select(["file_id", "verified", "bytes_downloaded", "duration_s", "attempts"]))

print()
print("=== Test asercji ===")
n_skipped = result_second.filter(pl.col("attempts") == 0).height
all_verified = result_second["verified"].all()
total_bytes_second = result_second["bytes_downloaded"].sum()

print(f"Plików pominiętych (attempts=0): {n_skipped}/{result_second.height}")
print(f"Wszystkie verified: {all_verified}")
print(f"Bytes ściągnięte powtórnie: {total_bytes_second} (powinno być 0)")


=== Drugie uruchomienie download_files (skip_existing=True default) ===


Pobieranie z GDC: 100%|█████████████████████████| 3/3 [00:00<00:00, 94.35plik/s]


=== Drugi raport ===
shape: (3, 5)
┌─────────────────────────────────┬──────────┬──────────────────┬────────────┬──────────┐
│ file_id                         ┆ verified ┆ bytes_downloaded ┆ duration_s ┆ attempts │
│ ---                             ┆ ---      ┆ ---              ┆ ---        ┆ ---      │
│ str                             ┆ bool     ┆ i64              ┆ f64        ┆ i64      │
╞═════════════════════════════════╪══════════╪══════════════════╪════════════╪══════════╡
│ 140633c2-a8e9-4a2d-a044-082cd0… ┆ true     ┆ 0                ┆ 0.0        ┆ 0        │
│ 15a57d06-841c-473e-b89e-389f1d… ┆ true     ┆ 0                ┆ 0.0        ┆ 0        │
│ 5ecec8fc-845f-43cf-82c3-73dea8… ┆ true     ┆ 0                ┆ 0.0        ┆ 0        │
└─────────────────────────────────┴──────────┴──────────────────┴────────────┴──────────┘

=== Test asercji ===
Plików pominiętych (attempts=0): 3/3
Wszystkie verified: True
Bytes ściągnięte powtórnie: 0 (powinno być 0)


## 7. KRYTYCZNY test integracji — pobrany plik → `parse_star_counts`

Czy pobrany przez API plik STAR Counts faktycznie parsuje się przez istniejący parser? Jeśli tak, pełna pętla API działa - od zapytania, przez pobieranie, po przekształcenie do polars DataFrame gotowego do dalszej analizy.


In [12]:
print("=== Wybieram pierwszy pobrany plik ===")
first_file = Path(result.filter(pl.col("verified"))["local_path"][0])
print(f"Plik: {first_file.name}")
print(f"Rozmiar: {first_file.stat().st_size / 1024**2:.1f} MB")
print()

print("=== Parsuję przez parse_star_counts ===")
star_df = parse_star_counts(first_file)
print(f"Wynik: {star_df.height} genów x {star_df.width} kolumn")
print()
print("Pierwsze 5 wierszy:")
print(star_df.head(5))
print()
print(f"Sprawdzenie: czy mamy 60660 genów (GENCODE v36)? {'TAK' if star_df.height == 60660 else 'NIE'}")


=== Wybieram pierwszy pobrany plik ===
Plik: b7f29b8c-08a8-4781-ba33-5a7b6b02ad23.rna_seq.augmented_star_gene_counts.tsv
Rozmiar: 4.0 MB

=== Parsuję przez parse_star_counts ===
Wynik: 60660 genów x 9 kolumn

Pierwsze 5 wierszy:
shape: (5, 9)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ gene_id   ┆ gene_name ┆ gene_type ┆ unstrande ┆ … ┆ stranded_ ┆ tpm_unstr ┆ fpkm_unst ┆ fpkm_uq_ │
│ ---       ┆ ---       ┆ ---       ┆ d         ┆   ┆ second    ┆ anded     ┆ randed    ┆ unstrand │
│ str       ┆ str       ┆ str       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ed       │
│           ┆           ┆           ┆ i64       ┆   ┆ i64       ┆ f64       ┆ f64       ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ ENSG00000 ┆ TSPAN6    ┆ protein_c ┆ 1569      ┆ 

## 8. Wnioski

Po przejściu przez ten notebook wiesz:

1. **Endpoint `/files` działa** — domyślny filtr (TCGA-LUAD + STAR Counts) zwraca poprawne metadane
2. **Endpoint `/cases` działa** — clinical w formacie identycznym z portalem
3. **Integracja `cases → TSV → parse_clinical` jest bezstratna** — pipeline jest niezależny od ręcznego pobierania
4. **`download_files` ściąga z weryfikacją MD5** — bezpieczne, audytowalne pobieranie
5. **Idempotentność działa** — można bezpiecznie wznawiać przerwane pobieranie
6. **Integracja `download → parse_star_counts` działa** — pobrany plik jest pełnoprawnym wejściem do reszty pipeline'u

**Co dalej:**

Brakuje jeszcze jednego elementu opcji B — komendy CLI `luad-huba download` spinającej wszystko w jedno polecenie. Po jej dodaniu repo będzie w pełni samowystarczalne: `git clone + luad-huba download = pełna kohorta TCGA-LUAD lokalnie`.

Jeśli wszystkie sekcje tego notebooka przeszły bez błędu — można bezpiecznie zaprojektować i zaimplementować CLI download (commit #4).
